### ЗАДАЧА: Распределение доставок между курьерами

Логистическая команда получает пакет заявок на доставки, которые нужно распределить между курьерами на электровелосипедах.
Нужно собрать систему, которая:
- принимает корректные доставки,
- отбрасывает неправильные или небезопасные заявки,
- уменьшает доступный заряд после успешно назначенного маршрута,
- ведёт отдельный журнал ошибок,
- помогает понять, какой курьер был загружен дольше всех и какому клиенту доставили самый большой вес.

In [ ]:
from dataclasses import dataclass
from typing import Optional


couriers = {
    'CR-1': {'zone': 'north', 'charge_min': 40, 'max_weight': 3.0},
    'CR-2': {'zone': 'south', 'charge_min': 30, 'max_weight': 2.0},
    'CR-3': {'zone': 'north', 'charge_min': 55, 'max_weight': 5.0},
}

# rows: delivery_id|courier_id|client|weight_kg|route_min
rows = [
    'DL-100|CR-1|Clinic|1.5|12',
    'DL-101|CR-2|Cafe|2.5|10',
    'DL-102|CR-9|Lab|1.0|8',
    'DL-103|CR-1|Shop|0|6',
    'DL-104|CR-3|Village|3.5|60',
    'DL-100|CR-3|Clinic|1.0|10',
    'DL-105|CR-3|School|2.0|20',
    'DL-106|CR-2|Pharmacy|1.0|15',
]


class DeliveryError(Exception):
    pass


class RowFormatError(DeliveryError):
    pass


class CourierNotFoundError(DeliveryError):
    pass


class WeightError(DeliveryError):
    pass


class RouteTimeError(DeliveryError):
    pass


class WeightLimitError(DeliveryError):
    pass


class ChargeReserveError(DeliveryError):
    pass


class DuplicateDeliveryError(DeliveryError):
    pass


@dataclass(order=True)
class Delivery:
    route_min: int
    delivery_id: str
    courier_id: str
    client: str
    weight_kg: float


class Courier:
    def __init__(self, courier_id, zone, charge_min, max_weight):
        # TODO: сохранить courier_id, zone, charge_min, max_weight
        self.courier_id = courier_id
        self.zone = zone
        self.charge_min = charge_min
        self.max_weight = max_weight
        # TODO: создать список deliveries
        self.deliveries = []

    def charge_left(self):
        # TODO: вернуть текущий остаток заряда в минутах
        used_charge = sum(delivery.route_min for delivery in self.deliveries)
        return self.charge_min - used_charge

    def total_route_time(self):
        # TODO: вернуть сумму route_min по self.deliveries
        return sum(delivery.route_min for delivery in self.deliveries)

    def total_weight(self):
        # TODO: вернуть сумму weight_kg по self.deliveries
        return sum(delivery.weight_kg for delivery in self.deliveries)

    def assign(self, delivery):
        # TODO: если delivery.weight_kg > self.max_weight -> raise WeightLimitError(...)
        if delivery.weight_kg > self.max_weight:
            raise WeightLimitError("Вес посылки больше максимального переносимого веса")
        # TODO: посчитать charge_after = charge_left() - delivery.route_min
        charge_after = self.charge_left() - delivery.route_min
        # TODO: если charge_after < 5 -> raise ChargeReserveError(...)
        if charge_after < 5:
            raise ChargeReserveError("Нехватает заряда для доставки")
        # TODO: добавить delivery в self.deliveries
        self.deliveries.append(delivery)
        # TODO: отсортировать self.deliveries
        self.deliveries.sort(key=lambda x: x.route_min)


class CourierDispatchService:
    def __init__(self, couriers):
        # TODO: создать couriers вида courier_id -> Courier(...)
        self.couriers: dict[str, Courier] = {}
        for courier_id, delivery in couriers.items():
            self.couriers[courier_id] = Courier(courier_id = courier_id, zone = delivery["zone"], charge_min = delivery["charge_min"], max_weight = delivery["max_weight"])
        # TODO: создать списки accepted и errors
        self.accepted = []
        self.errors = []
        # TODO: создать множество processed_ids
        self.processed_ids = set()

    def parse_delivery(self, row):
        # TODO: split по '|'
        parts = row.split("|")
        # TODO: ожидать 5 частей: delivery_id, courier_id, client, weight_raw, route_raw
        delivery_id, courier_id, client, weight_kg, route_min = parts
        # TODO: если частей не 5 -> raise RowFormatError(...)
        if len(parts) != 5:
            raise RowFormatError("Список дожен состоять из 5 частей")
        # TODO: проверить, что courier_id существует
        if courier_id not in self.couriers:
            raise CourierNotFoundError("Курьер с таким id не найден")
        # TODO: weight_raw преобразовать в float
        try:
            weight_kg = float(weight_kg)
        except ValueError as exc:
            raise WeightError("Вес посылки должен быть числом") from exc
        # TODO: route_raw преобразовать в int
        try:
            route_min = int(route_min)
        except ValueError as exc:
            raise RouteTimeError("Время доставки должно быть числом") from exc
        # TODO: ошибки преобразования поднимать через WeightError / RouteTimeError с raise ... from exc
        # TODO: если weight_kg <= 0 -> raise WeightError(...)
        if weight_kg <= 0:
            raise WeightError("Вес посылки не может быть отрицательным или равным 0")
        # TODO: если route_min <= 0 -> raise RouteTimeError(...)
        if route_min <= 0:
            raise RouteTimeError("Время доставки не может быть отрицательным или равным 0")
        # TODO: вернуть Delivery(...)
        return Delivery(delivery_id=delivery_id, courier_id=courier_id, client=client, weight_kg=weight_kg, route_min=route_min)

    def submit(self, row):
        # TODO: внутри try вызвать parse_delivery(row)
        try:
            delivery = self.parse_delivery(row)
        # TODO: если delivery.delivery_id уже в processed_ids -> raise DuplicateDeliveryError(...)
            if delivery.delivery_id in self.processed_ids:
                raise DuplicateDeliveryError("Такой id доставки уже существует")
        # TODO: передать delivery в couriers[delivery.courier_id].assign(delivery)
            self.couriers[delivery.courier_id].assign(delivery)
        # TODO: после успеха обновить processed_ids и accepted
            self.processed_ids.add(delivery.delivery_id)
            self.accepted.append(delivery)
        # TODO: DeliveryError сохранить в errors как (row, error_type, message)
        except DeliveryError as exc:
            self.errors.append((row, type(exc).__name__, str(exc)))

    def load(self, rows):
        # TODO: вызвать submit(row) для каждой строки
        for row in rows:
            self.submit(row)

    def client_weights(self):
        # TODO: собрать dict вида client -> total_weight_kg
        client_totals = {}
        for delivery in self.accepted:
            client_totals[delivery.client] = client_totals.get(delivery.client, 0) + delivery.weight_kg
        return client_totals

    def top_client(self):
        # TODO: использовать client_weights()
        totals = self.client_weights()
        if not totals:
            return None
        # TODO: вернуть tuple(client, weight_kg) с максимумом
        return max(totals.items(), key=lambda x: x[1])

    def busiest_courier(self):
        # TODO: найти курьера с максимумом total_route_time()
        courier = max(self.couriers.values(), key=lambda x: x.total_route_time())
        # TODO: вернуть tuple(courier_id, total_route_time)
        return (courier.courier_id, courier.total_route_time())

    def low_charge_couriers(self, threshold=15):
        # TODO: вернуть список tuple(courier_id, charge_left)
        # TODO: включать только курьеров, у которых charge_left() <= threshold
        result = []
        for courier in self.couriers.values():
            charge_left = courier.charge_left()
            if charge_left <= threshold:
                result.append((courier.courier_id, charge_left))
        return result
       
    def find_delivery(self, delivery_id) -> Optional[Delivery]:
        # TODO: пройтись по всем курьерам и их доставкам
        for delivery in self.accepted:
        # TODO: если delivery.delivery_id совпал -> вернуть объект Delivery
            if delivery.delivery_id == delivery_id:
                return delivery
        # TODO: если доставка не найдена -> вернуть None
        return None

service = CourierDispatchService(couriers)

# TODO: загрузить rows через service.load(rows)
service.load(rows)
# TODO: вывести принятые доставки
if service.accepted:
    print("Принятые заявки", len(service.accepted), "шт:")
    for accept in service.accepted:
        print("-", accept)
else:
    print("Принятых заявок не найдено")
# TODO: вывести ошибки
if service.errors:
    print("Ошибки", len(service.errors), "шт:")
    for error in service.errors:
        print("-", error)
else:
    print("Ошибок не найдено")
# TODO: вывести по каждому курьеру deliveries, total_route_time и charge_left
print("Информация по каждому курьеру:")
for courier_id, item in service.couriers.items():
    print(f"Курьер {courier_id}:")
    for delivery in item.deliveries:
        print(f"- {delivery}")
    print(f"- Общее время маршрутов: {item.total_route_time()} мин")
    print(f"- Остаток заряда: {item.charge_left()} мин")
# TODO: вывести top_client()
top_client = service.top_client()
if top_client:
    print(f"У клиента '{top_client[0]}' самая большая доставка = {top_client[1]} кг")
else:
    print("Клиентов не найдено")
# TODO: вывести busiest_courier()
busiest_courier = service.busiest_courier()
if busiest_courier[1] != 0:
    print(f"У курьера '{busiest_courier[0]}' самая долгая доставка = {busiest_courier[1]} мин")
else:
    print("Курьеров не найдено")
# TODO: вывести low_charge_couriers()
low_charge_couriers = service.low_charge_couriers()
if low_charge_couriers:
    for courier, min in low_charge_couriers:
        print(f"У курьера '{courier}' самый низкий заряд = {min} мин")
else:
    print("Курьер с низким зарядом не найден")
# TODO: вывести find_delivery('DL-105')
find_delivery = service.find_delivery("DL-105")
if find_delivery:
    print("Найденая заявка:")
    print("-", find_delivery)
else:
    print("В принятых заявках такой заявки не найдено")     

Принятые заявки 3 шт:
- Delivery(route_min=12, delivery_id='DL-100', courier_id='CR-1', client='Clinic', weight_kg=1.5)
- Delivery(route_min=20, delivery_id='DL-105', courier_id='CR-3', client='School', weight_kg=2.0)
- Delivery(route_min=15, delivery_id='DL-106', courier_id='CR-2', client='Pharmacy', weight_kg=1.0)
Ошибки 5 шт:
- ('DL-101|CR-2|Cafe|2.5|10', 'WeightLimitError', 'Вес посылки больше максимального переносимого веса')
- ('DL-102|CR-9|Lab|1.0|8', 'CourierNotFoundError', 'Курьер с таким id не найден')
- ('DL-103|CR-1|Shop|0|6', 'WeightError', 'Вес посылки не может быть отрицательным или равным 0')
- ('DL-104|CR-3|Village|3.5|60', 'ChargeReserveError', 'Нехватает заряда для доставки')
- ('DL-100|CR-3|Clinic|1.0|10', 'DuplicateDeliveryError', 'Такой id доставки уже существует')
Информация по каждому курьеру:
Курьер CR-1:
- Delivery(route_min=12, delivery_id='DL-100', courier_id='CR-1', client='Clinic', weight_kg=1.5)
- Общее время маршрутов: 12 мин
- Остаток заряда: 28 мин
Кур